# Qwen3.5-0.8B + FlyFFN-v3 — ALL FFNs replaced

This notebook tests **FlyFFN-v3** on **Qwen/Qwen3.5-0.8B**.

**Main v3 change:** all 24 Qwen FFN blocks are replaced by FlyFFN modules. There are **no dense FFN anchor layers**. Qwen3.5's Gated DeltaNet and full-attention token mixers remain untouched.

The conversion is still progressive and quality-gated: every FFN is structurally FlyFFN from the start, while each layer group may remain closer to dense behavior if a more aggressive routing stage violates the CE quality threshold.

After training the notebook runs FastEval, dual multi-turn chat, creates a verified standalone checkpoint, reloads it without base-model weights, and can upload it to Hugging Face.


In [ ]:
#@title 1. Update repository, install current Transformers, and syntax-check
import pathlib, subprocess, sys
REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers','accelerate','datasets','huggingface_hub','safetensors','ipywidgets','pandas','requests','tqdm'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)], check=True)
SRC_DIR = REPO_DIR/'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print('✓ Added to current kernel path:', SRC_DIR)
for p in [REPO_DIR/'src'/'tinycenn_lm'/'qwen35_flyffn_v3.py', REPO_DIR/'src'/'tinycenn_lm'/'qwen35_standalone.py', REPO_DIR/'scripts'/'run_qwen35_flyffn_v3.py']:
    subprocess.run([sys.executable,'-m','py_compile',str(p)], check=True)
print('✓ Qwen3.5 FlyFFN-v3 module and runner syntax OK')
import importlib.util
_spec=importlib.util.spec_from_file_location('qwen35_runner_preflight', REPO_DIR/'scripts'/'run_qwen35_flyffn_v3.py')
_runner=importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_runner)
assert hasattr(_runner.base.base,'set_seed') and hasattr(_runner.base.base,'choose_dtype')
assert all(hasattr(_runner.base,x) for x in ['stage_schedule','calibrate_progressively','global_train','dense_equivalence','v2_state','load_v2_state'])
from tinycenn_lm.qwen35_flyffn_v3 import FlyFFNV3Config, replace_all_ffns_with_fly_v3, assert_qwen35_flyffn_v3
from tinycenn_lm.qwen35_standalone import export_standalone, load_standalone, upload_standalone
print('✓ Runtime helper + standalone export wiring OK')
print('Ready:', REPO_DIR)


In [ ]:
#@title 2. Configuration — FlyFFN-v3 ALL FFNs
RUN_MODE = 'quick' #@param ['quick','strong']
SEQ_LEN = 128 #@param {type:'integer'}
BATCH_SIZE = 1 #@param {type:'integer'}
FLY_NODES = 256 #@param {type:'integer'}
ROUTER_RANK = 96 #@param {type:'integer'}
MAX_EDGES = 2048 #@param {type:'integer'}
NUM_SHARDS = 8 #@param {type:'integer'}
GRAPH_STEPS = 1 #@param {type:'integer'}
GRAPH_MIX_INIT = 0.50 #@param {type:'number'}
MAX_CE_GAP = 0.30 #@param {type:'number'}
RUN_REWIRED_CONTROL = False #@param {type:'boolean'}

OUTPUT_DIR = REPO_DIR/'results'/'flyffn_v3_qwen35_08b'

SAVE_STANDALONE = True #@param {type:'boolean'}
UPLOAD_TO_HF = True #@param {type:'boolean'}
HF_REPO_ID = "vtava/Qwen35-0.8B-FlyFFN-v3-AllFFN" #@param {type:'string'}
HF_PRIVATE = False #@param {type:'boolean'}

print('Base: Qwen/Qwen3.5-0.8B')
print('v3 structure: ALL 24 FFNs → FlyFFN-v3')
print('Dense anchors: 0')
print('Qwen token mixers: unchanged')
print('Progressive routing target: 8→6→4→4→3→2')
print('Quality gate CE gap <=', MAX_CE_GAP)
print('Rewired control:', RUN_REWIRED_CONTROL)
print('Standalone export:', SAVE_STANDALONE)
print('Upload to HF:', UPLOAD_TO_HF, '| repo:', HF_REPO_ID)


In [ ]:
#@title 3. Train / evaluate Qwen3.5 FlyFFN-v3 — live output
import os, subprocess, sys
cmd=[sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_flyffn_v3.py'),
     '--run-mode',RUN_MODE,'--seq-len',str(SEQ_LEN),'--batch-size',str(BATCH_SIZE),
     '--fly-nodes',str(FLY_NODES),'--router-rank',str(ROUTER_RANK),'--max-edges',str(MAX_EDGES),
     '--num-shards',str(NUM_SHARDS),'--graph-steps',str(GRAPH_STEPS),'--graph-mix-init',str(GRAPH_MIX_INIT),
'--max-ce-gap',str(MAX_CE_GAP),'--output-dir',str(OUTPUT_DIR)]
if RUN_REWIRED_CONTROL: cmd.append('--rewired')
if SAVE_STANDALONE:
    cmd += ['--standalone-dir', str(OUTPUT_DIR/'standalone')]
if UPLOAD_TO_HF:
    if not HF_REPO_ID.strip():
        raise ValueError('HF_REPO_ID is empty')
    from huggingface_hub import get_token, notebook_login
    token = os.environ.get('HF_TOKEN')
    try:
        from google.colab import userdata
        if not token:
            token = userdata.get('HF_TOKEN')
    except Exception:
        pass
    if not token:
        token = get_token()
    if not token:
        notebook_login()
        token = get_token()
    if not token:
        raise RuntimeError('Hugging Face login failed')
    os.environ['HF_TOKEN'] = token
    cmd += ['--upload-hf', '--hf-repo-id', HF_REPO_ID]
    if HF_PRIVATE:
        cmd.append('--hf-private')
print('='*100)
print('Qwen3.5-0.8B / FlyFFN-v3 progressive FFN experiment')
print('Command:', ' '.join(cmd)); print('='*100)
env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env['TQDM_MININTERVAL']='1'
if UPLOAD_TO_HF and os.environ.get('HF_TOKEN'):
    env['HF_TOKEN'] = os.environ['HF_TOKEN']
p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
for line in iter(p.stdout.readline,''):
    print(line,end='',flush=True)
rc=p.wait(); print('\nFinished, exit code',rc)
if rc: raise subprocess.CalledProcessError(rc,cmd)


In [ ]:
#@title 4. Training results
import json, pandas as pd
from IPython.display import display
summary=pd.read_csv(OUTPUT_DIR/'summary.csv',index_col=0)
report=json.loads((OUTPUT_DIR/'report.json').read_text())
chat_samples=json.loads((OUTPUT_DIR/'chat_samples.json').read_text())
display(summary)
print('\nCHECKS')
print('Architecture:',report['architecture'])
print('Token mixers unchanged:',report['token_mixers_unchanged'])
print('FlyFFN-v3 layers:',report['flyffn_layers'])
print('Dense anchors:',report['dense_anchor_layers'],'(v3 target: 0)')
print('Dense-equivalence max logit diff:',report['dense_equivalence_biological_max_abs_logit_diff'])
print('Device:',report['device'],'| dtype:',report['dtype'])
print('\nKEY METRICS')
for k in ['fly_ce_gap_vs_qwen','fly_ppl_ratio_vs_qwen','parameter_ratio_fly_over_qwen','decode_speed_ratio_fly_over_qwen','biological_topology_ce_gain','biological_topology_ppl_gain_pct']:
    if k in report: print(k,':',report[k])
print('\nFINAL ROUTING')
for layer,state in report['biological_routing_schedule'].items():
    print(f'layer {int(layer):>2}: k={state["active_k"]}, mix={state["route_mix"]:.2f}')
print('\nTRAINING CHAT SAMPLES')
for x in chat_samples:
    print('='*90); print('USER:',x['prompt']); print('FLY:',x['reply'])


In [ ]:
#@title 5. Load original Qwen3.5 + trained FlyFFN-v3 for FastEval and chat
import gc, json, sys
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.qwen35_flyffn_v3 import FlyFFNV3Config, replace_all_ffns_with_fly_v3, assert_qwen35_flyffn_v3

BASE_MODEL='Qwen/Qwen3.5-0.8B'
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype=torch.bfloat16 if device.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if device.type=='cuda' else torch.float32)
tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL,use_fast=True)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
report=json.loads((OUTPUT_DIR/'report.json').read_text())
state=torch.load(OUTPUT_DIR/'biological_qwen35_flyffn_v3.pt',map_location='cpu',weights_only=True)
adj=state['flyffn_shared_graph.adjacency'].float()
c=report['config']
cfg=FlyFFNV3Config(fly_nodes=int(c['fly_nodes']),router_rank=int(c['router_rank']),num_shards=int(c['num_shards']),graph_steps=int(c['graph_steps']),graph_mix_init=float(c['graph_mix_init']))

def load_base():
    return AutoModelForCausalLM.from_pretrained(BASE_MODEL,dtype=dtype if device.type=='cuda' else torch.float32,low_cpu_mem_usage=True).to(device).eval()

print('Loading original Qwen3.5-0.8B ...')
qwen_model=load_base()
print('Loading FlyFFN-v3 Qwen3.5 ...')
fly_model=load_base(); replace_all_ffns_with_fly_v3(fly_model,cfg,adj)
inc=fly_model.load_state_dict(state,strict=False)
missing=[k for k in inc.missing_keys if '.mlp.' in k or k.startswith('flyffn_shared_graph.')]
if missing: raise RuntimeError('Missing FlyFFN-v3 keys: '+str(missing[:10]))
assert_qwen35_flyffn_v3(fly_model)
print('✓ Both models ready on',device,'|',dtype)


In [ ]:
#@title 6. FastEval — 50 items: MMLU-Pro / PIQA / MMMLU-DE (GPQA optional)
import os, random, time
import pandas as pd, torch
from datasets import load_dataset
from IPython.display import display
N=50; EVAL_BATCH=4; MAX_LENGTH=1024; SEED=2026
LETTERS=list('ABCDEFGHIJ')

def sample(ds,n,seed): return ds.shuffle(seed=seed).select(range(min(n,len(ds))))
def prompt_mc(q,opts,german=False):
    labels=LETTERS[:len(opts)]
    lead=('Wähle die richtige Antwort. Antworte nur mit dem Buchstaben.' if german else 'Choose the correct answer. Reply only with the answer letter.')
    lines=[lead,'',('Frage: ' if german else 'Question: ')+str(q),'']+[f'{a}. {o}' for a,o in zip(labels,opts)]+['',('Antwort:' if german else 'Answer:')]
    return '\n'.join(lines),labels

def chat_wrap(text):
    return tokenizer.apply_chat_template([{'role':'user','content':text}],tokenize=False,add_generation_prompt=True)

def label_ids(labels):
    out=[]
    for a in labels:
        choices=[tokenizer.encode(' '+a,add_special_tokens=False),tokenizer.encode(a,add_special_tokens=False)]
        one=next((x[0] for x in choices if len(x)==1),None)
        if one is None: raise RuntimeError(f'Answer label {a} is not one token: {choices}')
        out.append(one)
    return out

benches={}
ds=sample(load_dataset('TIGER-Lab/MMLU-Pro',split='test'),N,SEED)
benches['MMLU-Pro']=[{'prompt':prompt_mc(x['question'],list(x['options']))[0],'labels':prompt_mc(x['question'],list(x['options']))[1],'gold':int(x['answer_index'])} for x in ds]
ds=sample(load_dataset('regisss/piqa',split='validation'),N,SEED+1)
benches['PIQA']=[{'prompt':prompt_mc(x['goal'],[x['sol1'],x['sol2']])[0],'labels':['A','B'],'gold':int(x['label'])} for x in ds]
try: ds=load_dataset('openai/MMMLU','DE_DE',split='test')
except Exception: ds=load_dataset('openai/MMMLU',split='test')
ds=sample(ds,N,SEED+2); mm=[]
for x in ds:
    opts=[str(x[k]) for k in ['A','B','C','D']]; p,l=prompt_mc(str(x['Question']),opts,True); mm.append({'prompt':p,'labels':l,'gold':l.index(str(x['Answer']).strip().upper())})
benches['MMMLU-DE']=mm
HF_TOKEN=os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    if not HF_TOKEN: HF_TOKEN=userdata.get('HF_TOKEN')
except: pass
try:
    ds=sample(load_dataset('Idavidrein/gpqa','gpqa_diamond',split='train',token=HF_TOKEN),N,SEED+3); gp=[]
    for i,x in enumerate(ds):
        raw=[x['Correct Answer'],x['Incorrect Answer 1'],x['Incorrect Answer 2'],x['Incorrect Answer 3']]; order=list(range(4)); random.Random(SEED+10000+i).shuffle(order)
        opts=[raw[j] for j in order]; p,l=prompt_mc(x['Question'],opts); gp.append({'prompt':p,'labels':l,'gold':order.index(0)})
    benches['GPQA-Diamond']=gp
except Exception as e: print('GPQA skipped:',str(e)[:140])

@torch.inference_mode()
def run_eval(model,name):
    ans={}; model.eval(); tokenizer.padding_side='right'; tokenizer.truncation_side='left'
    print('\n'+'='*92+'\nMODEL:',name+'\n'+'='*92)
    for bn,items in benches.items():
        correct=done=0; t0=time.perf_counter()
        for s in range(0,len(items),EVAL_BATCH):
            batch=items[s:s+EVAL_BATCH]; texts=[chat_wrap(x['prompt']) for x in batch]
            enc=tokenizer(texts,return_tensors='pt',padding=True,truncation=True,max_length=MAX_LENGTH).to(device)
            last=enc.attention_mask.sum(1)-1
            out=model(**enc,use_cache=False,return_dict=True).logits.float()
            for j,x in enumerate(batch):
                ids=torch.tensor(label_ids(x['labels']),device=device); pred=int(out[j,int(last[j])][ids].argmax())
                correct+=pred==x['gold']; done+=1
            if done%10==0 or done==len(items): print(f'  {done:>2}/{len(items)} | correct={correct:>2} | acc={100*correct/done:5.1f}% | {done/max(time.perf_counter()-t0,1e-9):5.2f} q/s')
        ans[bn]={'correct':correct,'total':done,'accuracy':correct/done}; print(f'  DONE → {correct}/{done} = {100*correct/done:.1f}%')
    return ans

base_eval=run_eval(qwen_model,'Qwen3.5-0.8B')
fly_eval=run_eval(fly_model,'Qwen3.5 FlyFFN-v3')
rows=[]
for bn in benches:
    b,f=base_eval[bn],fly_eval[bn]; bp,fp=100*b['accuracy'],100*f['accuracy']
    rows.append({'Benchmark':bn,'Qwen correct':f"{b['correct']}/{b['total']}",'Qwen %':bp,'FlyFFN-v3 correct':f"{f['correct']}/{f['total']}",'FlyFFN-v3 %':fp,'Δ FlyFFN-v3':fp-bp})
fast_eval_df=pd.DataFrame(rows).set_index('Benchmark')
display(fast_eval_df.style.format({'Qwen %':'{:.1f}%','FlyFFN-v3 %':'{:.1f}%','Δ FlyFFN-v3':'{:+.1f}'}))
print(f"Macro: Qwen={fast_eval_df['Qwen %'].mean():.1f}% | Fly={fast_eval_df['FlyFFN-v3 %'].mean():.1f}% | Δ={fast_eval_df['Δ FlyFFN-v3'].mean():+.1f} points")
fast_eval_df.to_csv(OUTPUT_DIR/'fast_eval_50_qwen35_v3.csv')


In [ ]:
#@title 7. Interactive dual CHAT — original Qwen3.5 vs FlyFFN-v3
import time, html, torch, ipywidgets as widgets
from IPython.display import display, HTML

base_history=[]; fly_history=[]

def _reply(model,history,user_text,max_new=192):
    msgs=history+[{'role':'user','content':user_text}]
    text=tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    enc=tokenizer(text,return_tensors='pt').to(device)
    t=time.perf_counter()
    with torch.inference_mode():
        out=model.generate(**enc,max_new_tokens=max_new,do_sample=False,use_cache=True,pad_token_id=tokenizer.eos_token_id)
    dt=time.perf_counter()-t
    reply=tokenizer.decode(out[0,enc.input_ids.shape[1]:],skip_special_tokens=True).strip()
    return reply,dt

def ask_both(user_text):
    global base_history, fly_history
    b,bt=_reply(qwen_model,base_history,user_text); f,ft=_reply(fly_model,fly_history,user_text)
    base_history += [{'role':'user','content':user_text},{'role':'assistant','content':b}]
    fly_history  += [{'role':'user','content':user_text},{'role':'assistant','content':f}]
    return b,bt,f,ft

def reset_chat(*_):
    global base_history,fly_history
    base_history=[]; fly_history=[]; out.clear_output()
    with out: print('Chat reset ✓')

prompt=widgets.Textarea(value='Explain in simple terms how a sparse FFN can save computation, and give one possible risk.',description='You:',layout=widgets.Layout(width='100%',height='100px'))
ask=widgets.Button(description='Ask both models',button_style='success'); reset=widgets.Button(description='Reset chat')
out=widgets.Output()

def on_ask(_):
    q=prompt.value.strip()
    if not q: return
    with out:
        print('\n'+'='*100); print('USER:',q)
        b,bt,f,ft=ask_both(q)
        table=(f"<table style='width:100%;table-layout:fixed'><tr><th>Original Qwen3.5-0.8B ({bt:.2f}s)</th>"
               f"<th>FlyFFN-v3 ({ft:.2f}s)</th></tr><tr><td style='vertical-align:top;white-space:pre-wrap;padding:12px'>"
               f"{html.escape(b)}</td><td style='vertical-align:top;white-space:pre-wrap;padding:12px'>{html.escape(f)}</td></tr></table>")
        display(HTML(table))
    prompt.value=''

ask.on_click(on_ask); reset.on_click(reset_chat)
display(widgets.VBox([prompt,widgets.HBox([ask,reset]),out]))
print('Dual multi-turn chat ready. Each model keeps its own conversation history.')


In [ ]:
#@title 8. Verify / repair v3 standalone export and upload (NO retraining needed)
import json, os
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.qwen35_flyffn_v3 import FlyFFNV3Config, replace_all_ffns_with_fly_v3, assert_qwen35_flyffn_v3
from tinycenn_lm.qwen35_standalone import export_standalone, upload_standalone

STANDALONE_DIR = OUTPUT_DIR/'standalone'
manifest_path = STANDALONE_DIR/'standalone_manifest.json'
state_path = STANDALONE_DIR/'standalone_state.pt'

if not (manifest_path.exists() and state_path.exists()):
    print('Verified v3 standalone package not found — rebuilding from trained checkpoint...')
    report = json.loads((OUTPUT_DIR/'report.json').read_text())
    c = report['config']
    cfg = FlyFFNV3Config(
        fly_nodes=int(c['fly_nodes']),
        router_rank=int(c['router_rank']),
        num_shards=int(c['num_shards']),
        graph_steps=int(c['graph_steps']),
        graph_mix_init=float(c['graph_mix_init']),
    )

    if 'fly_model' in globals():
        export_model = fly_model
        export_tokenizer = tokenizer
        print('Using FlyFFN-v3 model already in notebook memory.')
    else:
        ckpt = torch.load(
            OUTPUT_DIR/'biological_qwen35_flyffn_v3.pt',
            map_location='cpu',
            weights_only=True,
        )
        adjacency = ckpt['flyffn_shared_graph.adjacency'].float()
        _device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        _dtype = torch.bfloat16 if _device.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if _device.type=='cuda' else torch.float32)

        export_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3.5-0.8B', use_fast=True)
        if export_tokenizer.pad_token_id is None:
            export_tokenizer.pad_token = export_tokenizer.eos_token

        export_model = AutoModelForCausalLM.from_pretrained(
            'Qwen/Qwen3.5-0.8B',
            dtype=_dtype if _device.type=='cuda' else torch.float32,
            low_cpu_mem_usage=True,
        ).to(_device)

        replace_all_ffns_with_fly_v3(export_model, cfg, adjacency)
        incompatible = export_model.load_state_dict(ckpt, strict=False)
        missing = [k for k in incompatible.missing_keys if '.mlp.' in k or k.startswith('flyffn_shared_graph.')]
        if missing:
            raise RuntimeError('Training checkpoint missing FlyFFN-v3 keys: '+str(missing[:20]))
        assert_qwen35_flyffn_v3(export_model)
        export_model.eval()
        print('Reconstructed FlyFFN-v3 from saved training checkpoint.')

    manifest = export_standalone(
        export_model,
        export_tokenizer,
        cfg.to_v2(len(export_model.model.layers)),
        STANDALONE_DIR,
        metadata=report,
        artifacts_dir=OUTPUT_DIR,
    )
    print('✓ Repaired v3 standalone package created')

    if UPLOAD_TO_HF:
        url = upload_standalone(
            STANDALONE_DIR,
            HF_REPO_ID,
            token=os.environ.get('HF_TOKEN'),
            private=HF_PRIVATE,
        )
        print('✓ Uploaded repaired v3 standalone:', url)

manifest = json.loads(manifest_path.read_text())
assert manifest.get('verified') is True
assert state_path.exists()

print('\n✓ FlyFFN-v3 standalone model verified')
print('All FFNs   : 24/24 FlyFFN-v3')
print('Dense anchors: 0')
print('State keys :', manifest['state_keys'])
print('FlyFFN keys:', manifest['flyffn_state_keys'])
print('Size       :', f"{manifest['state_bytes']/1024**3:.3f} GB")
if UPLOAD_TO_HF:
    print('Hugging Face:', f'https://huggingface.co/{HF_REPO_ID}')


In [ ]:
#@title 9. Fresh standalone reload + chat test (no base-model download)
import gc, torch
from tinycenn_lm.qwen35_standalone import load_standalone

# Free comparison models if they exist to avoid GPU OOM.
for _name in ['qwen_model', 'fly_model']:
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

standalone_model, standalone_tokenizer = load_standalone(
    OUTPUT_DIR / 'standalone',
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

messages=[{'role':'user','content':'Explain in simple terms what a neural network is.'}]
prompt=standalone_tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
inputs=standalone_tokenizer(prompt,return_tensors='pt').to(next(standalone_model.parameters()).device)

with torch.inference_mode():
    out=standalone_model.generate(
        **inputs,
        max_new_tokens=160,
        do_sample=False,
        use_cache=True,
        pad_token_id=standalone_tokenizer.eos_token_id,
    )

answer=standalone_tokenizer.decode(
    out[0,inputs.input_ids.shape[1]:],
    skip_special_tokens=True,
).strip()

print('✓ Fresh standalone reconstruction succeeded')
print('\nFlyFFN-v3 standalone answer:\n', answer)


In [ ]:
#@title 10. Sync latest FastEval/results to the Hugging Face model repo
import json, shutil, os
from pathlib import Path
from tinycenn_lm.qwen35_standalone import upload_standalone

STANDALONE_DIR = OUTPUT_DIR / 'standalone'

# If FastEval was run, include it in the model repository.
if (OUTPUT_DIR/'fast_eval_50_qwen35_v3.csv').exists():
    shutil.copy2(
        OUTPUT_DIR/'fast_eval_50_qwen35_v3.csv',
        STANDALONE_DIR/'fast_eval_50_qwen35_v3.csv'
    )
    print('✓ Added latest FastEval CSV')

if 'fast_eval_df' in globals():
    fast_eval_df.to_csv(STANDALONE_DIR/'fast_eval_50_qwen35_v3.csv')

if UPLOAD_TO_HF:
    url = upload_standalone(
        STANDALONE_DIR,
        HF_REPO_ID,
        token=os.environ.get('HF_TOKEN'),
        private=HF_PRIVATE,
    )
    print('✓ Hugging Face synchronized:', url)
else:
    print('UPLOAD_TO_HF=False — standalone model remains local at', STANDALONE_DIR)
